# 08 · Saving to a Database (Supabase) 🗄️

**Supabase** gives you a free Postgres database with a nice web dashboard.

## Step 0 — set up (5 min)
1. Sign in at **supabase.com** → **New project** (free, no credit card). Pick region **Singapore**.
2. **SQL Editor → New query** → paste everything from `supabase/setup.sql` → **Run**.
3. **Project Settings → API Keys** → copy the **Project URL** and a **Secret key** (`sb_secret_...`).
4. Add them to your `.env`:
```
SUPABASE_URL=https://xxxx.supabase.co
SUPABASE_SECRET_KEY=sb_secret_...
```

⚠️ The secret key can read **everything**. It only ever goes in `.env` or Vercel settings — never in your code, never on GitHub.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

if not os.path.exists("data"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
SUPABASE_URL = os.environ.get("SUPABASE_URL") or input("Supabase project URL: ")
SUPABASE_KEY = os.environ.get("SUPABASE_SECRET_KEY") or getpass("Supabase secret key: ")

In [ ]:
from supabase import create_client

db = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Connected!")

## 1. Insert a row

In [ ]:
db.table("messages").insert({
    "chat_id": 111,
    "user_name": "Mei Ling",
    "role": "user",
    "text": "one kopi siew dai please",
}).execute()

👉 Open **Table Editor → messages** in Supabase. Your row is there!

## 2. Read rows back

In [ ]:
rows = db.table("messages").select("*").eq("chat_id", 111).execute().data

for row in rows:
    print(row["user_name"], ":", row["text"])

## 3. Save & load a conversation ID
`upsert` = **insert** if it's new, **update** if it already exists.

In [ ]:
def save_conversation_id(chat_id, conversation_id):
    db.table("chats").upsert({"chat_id": chat_id, "conversation_id": conversation_id}).execute()


def get_conversation_id(chat_id):
    rows = db.table("chats").select("conversation_id").eq("chat_id", chat_id).execute().data
    if rows:
        return rows[0]["conversation_id"]
    return None


save_conversation_id(111, "conv_test_123")
print(get_conversation_id(111))
print(get_conversation_id(999))   # a chat we've never seen

## 4. Memory that survives restarts 🎉
Notebook 07's `chat()` — but the dictionary is replaced by the database.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

from agents import Agent, OpenAIConversationsSession, Runner

uncle = Agent(
    name="Kopi Uncle",
    instructions="You are a friendly kopitiam uncle. Keep replies short.",
    model=os.environ.get("OPENAI_MODEL") or "gpt-6-luna",
)


async def chat(chat_id, name, text):
    db.table("messages").insert({"chat_id": chat_id, "user_name": name, "role": "user", "text": text}).execute()

    session = OpenAIConversationsSession(conversation_id=get_conversation_id(chat_id))
    result = await Runner.run(uncle, text, session=session)
    save_conversation_id(chat_id, session.session_id)

    db.table("messages").insert({"chat_id": chat_id, "user_name": name, "role": "bot", "text": result.final_output}).execute()
    return result.final_output


print(await chat(333, "Ahmad", "Hi Uncle, I'm Ahmad. Teh-c for me."))

Now **restart the kernel** (⟳ at the top), run the setup cells again, and then:

In [ ]:
print(await chat(333, "Ahmad", "What's my usual, Uncle?"))

## 5. Demo-day leaderboard 🏆
Who chatted with your bot the most? Load the table into pandas.

In [ ]:
import pandas as pd

rows = db.table("messages").select("user_name, role").execute().data
df = pd.DataFrame(rows)

df[df["role"] == "user"].groupby("user_name").size().sort_values(ascending=False)

> ✅ The exact same functions live in `bot/memory.py`. When `SUPABASE_URL` is set, your bot uses the database automatically.

## ✏️ Exercises
1. Delete your test rows: `db.table("chats").delete().eq("chat_id", 111).execute()`
2. Count how many messages the **bot** has sent.
3. 🌶️ Challenge: in Supabase, add a `favourite_drink` column to `chats`, and write a tool so Uncle can remember it.

In [ ]:
# your code here